# S2 — Krigeage (processus gaussien) vs forêts, validation par blocs (mines)

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/inphb/ic2/notebooks/IC2_S2_krigeage_vs_foret.ipynb)
> Données : dossier `IC2_TP_donnees`. Bibliothèques : pandas, numpy, matplotlib, scikit-learn, torch, shap.

In [ ]:
# Configuration : Colab ou Jupyter local. Exécutez cette cellule en premier.
import os, shutil, urllib.request
DONNEES = ['geochimie_sols_spatial.csv']
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/inphb/ic2/data/'
for f in DONNEES:
    if os.path.exists(f):
        continue
    if os.path.exists(os.path.join('..', 'data', f)):
        shutil.copy(os.path.join('..', 'data', f), f)   # kit hors ligne / dépôt cloné
    else:
        urllib.request.urlretrieve(URL + f, f)            # Colab : téléchargement
    print('OK', f)


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold, GroupKFold
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
s = pd.read_csv('geochimie_sols_spatial.csv'); XY = s[['x_m', 'y_m']].values / 1000; z = s['log10_au_ppb'].values
COV = s[['unite_volc', 'ndvi', 'pente_deg']].values
blocs = (s['x_m'] // 750).astype(int) * 10 + (s['y_m'] // 750).astype(int)

## 1. Variogramme expérimental

In [ ]:
d = np.sqrt(((XY[:, None] - XY[None])**2).sum(-1)) * 1000; g = 0.5 * (z[:, None] - z[None])**2; iu = np.triu_indices(len(z), 1)
bins = np.arange(0, 1600, 80); h = (bins[:-1] + bins[1:]) / 2
gam = [g[iu][(d[iu] >= a) & (d[iu] < b_)].mean() for a, b_ in zip(bins[:-1], bins[1:])]
plt.plot(h, gam, 'o-'); plt.axhline(z.var(), ls='--'); plt.xlabel('h (m)'); plt.show()

## 2. Quatre modèles, deux protocoles

In [ ]:
def gp(): return GaussianProcessRegressor(ConstantKernel(0.5) * RBF(0.4, (0.05, 3)) + WhiteKernel(0.05), normalize_y=True)
def rf(): return RandomForestRegressor(300, min_samples_leaf=5, random_state=0, n_jobs=-1)
modeles = {'krigeage': (gp, XY), 'foret_xy': (rf, XY), 'foret_cov': (rf, COV), 'foret_tout': (rf, np.c_[XY, COV])}
for nom_cv, cv, grp in [('aléatoire', KFold(5, shuffle=True, random_state=0), None), ('blocs', GroupKFold(5), blocs)]:
    for nom, (mk, Xm) in modeles.items():
        sc = [r2_score(z[te], mk().fit(Xm[tr], z[tr]).predict(Xm[te])) for tr, te in cv.split(Xm, z, grp)]
        print(nom_cv, nom, round(np.mean(sc), 2))

## 3. Cartes d’estimation et d’incertitude

In [ ]:
m = gp().fit(XY, z); gx, gy = np.meshgrid(np.linspace(0, 3, 80), np.linspace(0, 3, 80))
mu, sd = m.predict(np.c_[gx.ravel(), gy.ravel()], return_std=True); print(m.kernel_)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5)); a = ax[0].contourf(gx, gy, mu.reshape(gx.shape), 20, cmap='YlOrRd'); plt.colorbar(a, ax=ax[0])
b_ = ax[1].contourf(gx, gy, sd.reshape(gx.shape), 20, cmap='Blues'); plt.colorbar(b_, ax=ax[1]); ax[1].scatter(XY[:, 0], XY[:, 1], s=2, c='k'); plt.show()

## Bonus : *regression kriging*
Kriger les résidus d’une forêt entraînée sur les covariables, additionner les deux prédictions, et comparer en blocs.